# Week 1 Exercise: Personal Technical Tutor

This notebook builds a small AI tutor for technical questions using OpenAI and Ollama.

The tutor is personalized for my learning style: friendly but concise answers, practical examples, and occasional references to Godot/game development, PHP, or DevOps when they help explain the concept.

In [ ]:
# imports

import os

from dotenv import load_dotenv
from openai import OpenAI


In [ ]:
# constants

MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"
OLLAMA_BASE_URL = "http://localhost:11434/v1"


In [ ]:
# set up environment

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is missing. Add it to your .env file.")

openai = OpenAI()
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


## Ask a technical question

Type over the question below whenever you want to use this as a study tool during the course.

In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what is the best AI workflow to use with Godot to make a retro style rpg game with pixel art
"""


In [ ]:
# personalized tutor prompt

system_prompt = """
You are Bruno's personal AI Engineering tutor.

Bruno likes answers that are friendly, direct, and succinct. He is learning AI engineering,
works with PHP, enjoys game development with Godot, and wants to learn more DevOps.

How to answer:
- Start with the practical meaning first.
- Keep the answer concise, but include enough detail to be useful.
- Use small code examples only when they clarify the idea.
- Mention Godot/game development, PHP, or DevOps only when the analogy genuinely helps.
- Point out common gotchas or edge cases.
- If the question is ambiguous, state the most likely interpretation and continue.
- Use markdown, but do not wrap the entire answer in a code block.
"""

example_question = """
What is a generator in Python?
"""

example_answer = """
A generator is a function or expression that produces values one at a time instead of building
a full list in memory.

The key idea is lazy execution: Python pauses the generator after each `yield` and resumes it
when the next value is requested.

Small example:

```python
def numbers():
    yield 1
    yield 2

for number in numbers():
    print(number)
```

This is useful when the sequence could be large, like streaming logs in a DevOps script or
loading game assets step by step instead of all at once.
"""

bad_answer_example = """
Generators are special iterable coroutine-like lazy objects that implement iterator protocols
and facilitate deferred execution semantics.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": example_question},
    {"role": "assistant", "content": example_answer},
    {
        "role": "user",
        "content": "Avoid answers like this because they are too abstract and jargon-heavy:\n"
        + bad_answer_example,
    },
    {
        "role": "assistant",
        "content": "Understood. I will explain concepts plainly and practically, with just enough technical detail.",
    },
    {"role": "user", "content": question},
]


In [ ]:
# Get gpt-4o-mini to answer, with streaming

stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True,
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)


In [ ]:
# Get Llama 3.2 to answer

response = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages,
)

print(response.choices[0].message.content)


## Notes

To try other local models, pull them with Ollama and change `MODEL_LLAMA`. For example, smaller machines may prefer `llama3.2:1b`.